In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
STEROIDS_PATH = '../datasets/anabolic_steroids.csv'
POWERLIFTING_PATH = '../datasets/openpowerlifting-2026-06-20/openpowerlifting-2026-06-20-a9718576.csv'

steroids = pd.read_csv(STEROIDS_PATH)
powerlifting = pd.read_csv(POWERLIFTING_PATH, low_memory=False)

In [3]:
powerlifting

,Name,Sex,Event,Equipment,Age,AgeClass,BirthYearClass,Division,BodyweightKg,WeightClassKg,...,Country,State,Federation,ParentFederation,Date,MeetCountry,MeetState,MeetTown,MeetName,Sanctioned
0,E.S. Denisenko,F,B,Raw,28.5,24-34,24-39,Open,67.3,NaN,...,Belarus,NaN,GSF-Belarus,NaN,2019-06-22,Belarus,NaN,Luninets,Bison Power Cup,Yes
1,I.S. Lebetskaya,F,B,Raw,43.5,40-44,40-49,Open,73.2,NaN,...,Belarus,NaN,GSF-Belarus,NaN,2019-06-22,Belarus,NaN,Luninets,Bison Power Cup,Yes
2,K. Yakimovich,F,B,Raw,26.5,24-34,24-39,Open,60.6,NaN,...,Belarus,NaN,GSF-Belarus,NaN,2019-06-22,Belarus,NaN,Luninets,Bison Power Cup,Yes
3,A.G. Golneva,F,B,Raw,19.5,20-23,19-23,Juniors 17-21,50.3,NaN,...,Belarus,NaN,GSF-Belarus,NaN,2019-06-22,Belarus,NaN,Luninets,Bison Power Cup,Yes
4,E.V. Marunevskaya,F,B,Raw,19.5,20-23,19-23,Juniors 17-21,63.7,NaN,...,Belarus,NaN,GSF-Belarus,NaN,2019-06-22,Belarus,NaN,Luninets,Bison Power Cup,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3954725,Ibrahim Sandogji,M,SBD,Raw,17.0,16-17,14-18,Sub-Juniors,64.4,66,...,Saudi Arabia,NaN,SSSC,IPF,2022-10-13,Saudi Arabia,NaN,Riyadh,The Kingdom Classic Powerlifting Championships,Yes
3954726,Feras Alwahhabi,M,SBD,Raw,17.0,16-17,14-18,Sub-Juniors,82.9,83,...,Saudi Arabia,NaN,SSSC,IPF,2022-10-13,Saudi Arabia,NaN,Riyadh,The Kingdom Classic Powerlifting Championships,Yes
3954727,Musab Alessa,M,SBD,Raw,17.0,16-17,14-18,Sub-Juniors,91.3,93,...,Saudi Arabia,NaN,SSSC,IPF,2022-10-13,Saudi Arabia,NaN,Riyadh,The Kingdom Classic Powerlifting Championships,Yes
3954728,Ali Alholan,M,SBD,Raw,18.0,18-19,14-18,Sub-Juniors,103.6,105,...,Saudi Arabia,NaN,SSSC,IPF,2022-10-13,Saudi Arabia,NaN,Riyadh,The Kingdom Classic Powerlifting Championships,Yes


In [4]:
powerlifting.columns

Index(['Name', 'Sex', 'Event', 'Equipment', 'Age', 'AgeClass',
       'BirthYearClass', 'Division', 'BodyweightKg', 'WeightClassKg',
       'Squat1Kg', 'Squat2Kg', 'Squat3Kg', 'Squat4Kg', 'Best3SquatKg',
       'Bench1Kg', 'Bench2Kg', 'Bench3Kg', 'Bench4Kg', 'Best3BenchKg',
       'Deadlift1Kg', 'Deadlift2Kg', 'Deadlift3Kg', 'Deadlift4Kg',
       'Best3DeadliftKg', 'TotalKg', 'Place', 'Dots', 'Wilks', 'Glossbrenner',
       'Goodlift', 'Tested', 'Country', 'State', 'Federation',
       'ParentFederation', 'Date', 'MeetCountry', 'MeetState', 'MeetTown',
       'MeetName', 'Sanctioned'],
      dtype='str')

In [5]:
# il dataset è enorme, rimuovo le colonne che non mi serviranno

clean_powerlifting = powerlifting[[
    'Sex', 'Age', 'BodyweightKg', 
    'Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg',
    'TotalKg', 'Place', 'Dots', 'Wilks', 'Tested', 'Federation', 'Date'
]].copy()

clean_powerlifting

,Sex,Age,BodyweightKg,Best3SquatKg,Best3BenchKg,Best3DeadliftKg,TotalKg,Place,Dots,Wilks,Tested,Federation,Date
0,F,28.5,67.3,NaN,NaN,NaN,NaN,DQ,NaN,NaN,NaN,GSF-Belarus,2019-06-22
1,F,43.5,73.2,NaN,90.0,NaN,90.0,1,88.80,86.89,NaN,GSF-Belarus,2019-06-22
2,F,26.5,60.6,NaN,45.0,NaN,45.0,2,49.57,49.79,NaN,GSF-Belarus,2019-06-22
3,F,19.5,50.3,NaN,35.0,NaN,35.0,2,43.67,44.76,NaN,GSF-Belarus,2019-06-22
4,F,19.5,63.7,NaN,42.5,NaN,42.5,1,45.40,45.26,NaN,GSF-Belarus,2019-06-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3954725,M,17.0,64.4,177.5,120.0,215.0,512.5,1,409.03,410.73,Yes,SSSC,2022-10-13
3954726,M,17.0,82.9,212.5,112.5,230.0,555.0,1,374.93,370.73,Yes,SSSC,2022-10-13
3954727,M,17.0,91.3,242.5,130.0,257.5,630.0,1,404.46,399.31,Yes,SSSC,2022-10-13
3954728,M,18.0,103.6,215.0,120.0,255.0,590.0,1,357.78,354.25,Yes,SSSC,2022-10-13


## Place: DD significa Doping Disqualified
Quindi ci possono essere atleti che usano steroidi anche nei natural

## Isolo solo gli anni, senza il resto della data

In [6]:
clean_powerlifting['Year'] = clean_powerlifting['Date'].str[0:4].astype(int)
clean_powerlifting

,Sex,Age,BodyweightKg,Best3SquatKg,Best3BenchKg,Best3DeadliftKg,TotalKg,Place,Dots,Wilks,Tested,Federation,Date,Year
0,F,28.5,67.3,NaN,NaN,NaN,NaN,DQ,NaN,NaN,NaN,GSF-Belarus,2019-06-22,2019
1,F,43.5,73.2,NaN,90.0,NaN,90.0,1,88.80,86.89,NaN,GSF-Belarus,2019-06-22,2019
2,F,26.5,60.6,NaN,45.0,NaN,45.0,2,49.57,49.79,NaN,GSF-Belarus,2019-06-22,2019
3,F,19.5,50.3,NaN,35.0,NaN,35.0,2,43.67,44.76,NaN,GSF-Belarus,2019-06-22,2019
4,F,19.5,63.7,NaN,42.5,NaN,42.5,1,45.40,45.26,NaN,GSF-Belarus,2019-06-22,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3954725,M,17.0,64.4,177.5,120.0,215.0,512.5,1,409.03,410.73,Yes,SSSC,2022-10-13,2022
3954726,M,17.0,82.9,212.5,112.5,230.0,555.0,1,374.93,370.73,Yes,SSSC,2022-10-13,2022
3954727,M,17.0,91.3,242.5,130.0,257.5,630.0,1,404.46,399.31,Yes,SSSC,2022-10-13,2022
3954728,M,18.0,103.6,215.0,120.0,255.0,590.0,1,357.78,354.25,Yes,SSSC,2022-10-13,2022


In [7]:
clean_powerlifting = clean_powerlifting.drop('Date', axis='columns')

In [8]:
clean_powerlifting = clean_powerlifting.dropna(subset=['Dots', 'Year', 'TotalKg'])
clean_powerlifting

,Sex,Age,BodyweightKg,Best3SquatKg,Best3BenchKg,Best3DeadliftKg,TotalKg,Place,Dots,Wilks,Tested,Federation,Year
1,F,43.5,73.2,NaN,90.0,NaN,90.0,1,88.80,86.89,NaN,GSF-Belarus,2019
2,F,26.5,60.6,NaN,45.0,NaN,45.0,2,49.57,49.79,NaN,GSF-Belarus,2019
3,F,19.5,50.3,NaN,35.0,NaN,35.0,2,43.67,44.76,NaN,GSF-Belarus,2019
4,F,19.5,63.7,NaN,42.5,NaN,42.5,1,45.40,45.26,NaN,GSF-Belarus,2019
5,M,30.5,82.0,NaN,132.5,NaN,132.5,2,90.07,89.09,NaN,GSF-Belarus,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3954725,M,17.0,64.4,177.5,120.0,215.0,512.5,1,409.03,410.73,Yes,SSSC,2022
3954726,M,17.0,82.9,212.5,112.5,230.0,555.0,1,374.93,370.73,Yes,SSSC,2022
3954727,M,17.0,91.3,242.5,130.0,257.5,630.0,1,404.46,399.31,Yes,SSSC,2022
3954728,M,18.0,103.6,215.0,120.0,255.0,590.0,1,357.78,354.25,Yes,SSSC,2022


In [9]:
is_natural = (clean_powerlifting['Place'] != 'DD') & (clean_powerlifting['Tested'] == 'Yes')
clean_powerlifting['Group'] = np.where(is_natural, 'Natural', 'Unknown')

clean_powerlifting

,Sex,Age,BodyweightKg,Best3SquatKg,Best3BenchKg,Best3DeadliftKg,TotalKg,Place,Dots,Wilks,Tested,Federation,Year,Group
1,F,43.5,73.2,NaN,90.0,NaN,90.0,1,88.80,86.89,NaN,GSF-Belarus,2019,Unknown
2,F,26.5,60.6,NaN,45.0,NaN,45.0,2,49.57,49.79,NaN,GSF-Belarus,2019,Unknown
3,F,19.5,50.3,NaN,35.0,NaN,35.0,2,43.67,44.76,NaN,GSF-Belarus,2019,Unknown
4,F,19.5,63.7,NaN,42.5,NaN,42.5,1,45.40,45.26,NaN,GSF-Belarus,2019,Unknown
5,M,30.5,82.0,NaN,132.5,NaN,132.5,2,90.07,89.09,NaN,GSF-Belarus,2019,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3954725,M,17.0,64.4,177.5,120.0,215.0,512.5,1,409.03,410.73,Yes,SSSC,2022,Natural
3954726,M,17.0,82.9,212.5,112.5,230.0,555.0,1,374.93,370.73,Yes,SSSC,2022,Natural
3954727,M,17.0,91.3,242.5,130.0,257.5,630.0,1,404.46,399.31,Yes,SSSC,2022,Natural
3954728,M,18.0,103.6,215.0,120.0,255.0,590.0,1,357.78,354.25,Yes,SSSC,2022,Natural


In [10]:
clean_powerlifting.to_csv('../datasets/cleaned_openpowerlifting.csv', index=False)